# Train Neurodiverse Brain Model

Fine-tunes TRIBE v2 on autism fMRI data (ABIDE).

**Strategy**: Freeze the backbone (LLaMA + Wav2Vec + Transformer), only retrain the subject-specific output layers on ASD vs TD brain patterns.

**Runtime**: Set to `T4 GPU` before running.

After training, download the checkpoint and deploy to Azure VM.

In [ ]:
# Install dependencies (fix numpy compatibility first)
!git clone https://github.com/Ibrhimovic9989/tribeneuro.git
%cd tribeneuro
!pip install numpy==2.1.0 -q
!pip install scipy scikit-learn nilearn joblib -q
!pip install -e . -q

In [ ]:
# Login to HuggingFace
from huggingface_hub import login
login(token="PASTE_YOUR_HF_TOKEN_HERE")

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.mem_get_info(0)[1] / 1024**3:.1f} GB')

## Step 1: Download ABIDE Data + Compute Features

In [ ]:
import numpy as np
import pandas as pd
import logging
logging.basicConfig(level=logging.INFO)

# Download ABIDE (100 subjects: ~50 ASD, ~50 TD)
from nilearn.datasets import fetch_abide_pcp

dataset = fetch_abide_pcp(
    data_dir='./data/abide',
    pipeline='cpac',
    band_pass_filtering=True,
    global_signal_regression=False,
    derivatives=['func_preproc'],
    n_subjects=100,
)

phenotypic = pd.DataFrame(dataset.phenotypic)
phenotypic['func_path'] = [str(p) for p in dataset.func_preproc]
phenotypic['diagnosis'] = phenotypic['DX_GROUP'].map({1: 'ASD', 2: 'TD'})

print(f'Subjects: {len(phenotypic)}')
print(phenotypic.groupby('diagnosis').size())

## Step 2: Extract Brain Features

Project each subject's fMRI to the same surface space as TRIBE v2 (fsaverage5, 20,484 vertices) and compute functional connectivity.

In [ ]:
from nilearn import datasets as ni_datasets, maskers
from sklearn.model_selection import train_test_split

# Use Schaefer atlas (100 ROIs, 7 networks) for connectivity
atlas = ni_datasets.fetch_atlas_schaefer_2018(n_rois=100)

masker = maskers.NiftiLabelsMasker(
    labels_img=atlas.maps,
    standardize='zscore_sample',
    detrend=True,
    low_pass=0.1,
    high_pass=0.01,
    t_r=2.0,
)

# Extract connectivity features for each subject
features = []
labels = []
valid_idx = []

for i, row in phenotypic.iterrows():
    try:
        ts = masker.fit_transform(row['func_path'])
        conn = np.corrcoef(ts.T)
        conn = np.arctanh(np.clip(conn, -0.999, 0.999))
        np.fill_diagonal(conn, 0)
        # Upper triangle as feature vector
        upper = conn[np.triu_indices(100, k=1)]
        features.append(upper)
        labels.append(1 if row['diagnosis'] == 'ASD' else 0)
        valid_idx.append(i)
        print(f'  {len(features)}/{len(phenotypic)} - {row["diagnosis"]}', end='\r')
    except Exception as e:
        print(f'  Skip {i}: {e}')

X = np.array(features)
y = np.array(labels)
print(f'\nExtracted: {len(X)} subjects ({sum(y)} ASD, {len(y)-sum(y)} TD)')
print(f'Feature shape: {X.shape} (each = {100*99//2} connectivity pairs)')

## Step 3: Train the Neurodiverse Transform Model

This learns a **transform function** that converts neurotypical brain predictions to neurodiverse predictions.

Architecture:
- Input: TRIBE v2's NT brain prediction (20,484 vertices)
- Transform: Learned per-vertex scaling + shifting based on ASD connectivity patterns
- Output: Predicted ND brain activity (20,484 vertices)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class NeurodiverseTransform(nn.Module):
    """Learns to transform NT brain activity patterns to ND patterns.
    
    Uses ABIDE connectivity differences to learn per-network
    scaling and shifting of brain activity.
    """
    def __init__(self, n_vertices=20484, n_connectivity_features=4950, n_networks=7):
        super().__init__()
        
        # Connectivity encoder: learns which connectivity patterns matter
        self.conn_encoder = nn.Sequential(
            nn.Linear(n_connectivity_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        
        # Predicts per-vertex scale and shift
        self.scale_head = nn.Sequential(
            nn.Linear(128, 1024),
            nn.ReLU(),
            nn.Linear(1024, n_vertices),
            nn.Sigmoid(),  # scale between 0 and 2
        )
        self.shift_head = nn.Sequential(
            nn.Linear(128, 1024),
            nn.ReLU(),
            nn.Linear(1024, n_vertices),
            nn.Tanh(),  # shift between -1 and 1
        )
        
        # ASD/TD classifier head (auxiliary task)
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )
        
    def forward(self, connectivity):
        """Returns scale, shift, and ASD probability."""
        encoded = self.conn_encoder(connectivity)
        scale = self.scale_head(encoded) * 2  # range [0, 2]
        shift = self.shift_head(encoded) * 0.3  # range [-0.3, 0.3]
        asd_prob = self.classifier(encoded)
        return scale, shift, asd_prob
    
    def transform_brain(self, nt_prediction, connectivity):
        """Transform NT brain prediction to ND prediction."""
        scale, shift, _ = self.forward(connectivity)
        nd_prediction = nt_prediction * scale + shift
        return nd_prediction

print('Model defined')
model = NeurodiverseTransform()
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Split data
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

train_ds = TensorDataset(
    torch.FloatTensor(X_train),
    torch.FloatTensor(y_train),
)
val_ds = TensorDataset(
    torch.FloatTensor(X_val),
    torch.FloatTensor(y_val),
)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)

print(f'Train: {len(X_train)} ({sum(y_train)} ASD)')
print(f'Val: {len(X_val)} ({sum(y_val)} ASD)')

In [ ]:
# Train
device = torch.device('cuda')
model = NeurodiverseTransform().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
criterion = nn.BCELoss()

# Also compute mean ASD and TD connectivity for the reconstruction loss
asd_mean_conn = torch.FloatTensor(X[y == 1].mean(axis=0)).to(device)
td_mean_conn = torch.FloatTensor(X[y == 0].mean(axis=0)).to(device)
conn_diff = asd_mean_conn - td_mean_conn

best_val_acc = 0
best_epoch = 0

for epoch in range(100):
    # Train
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        scale, shift, asd_prob = model(batch_x)
        
        # Classification loss
        cls_loss = criterion(asd_prob.squeeze(), batch_y)
        
        # Connectivity reconstruction: ASD subjects should produce
        # scale/shift that reflects the known ASD-TD differences
        asd_mask = batch_y == 1
        if asd_mask.sum() > 0:
            # ASD subjects should have scale != 1 in regions that differ
            asd_scale = scale[asd_mask].mean(dim=0)
            scale_reg = ((asd_scale - 1.0).abs().mean() - 0.1).clamp(min=0)
        else:
            scale_reg = torch.tensor(0.0).to(device)
        
        td_mask = batch_y == 0
        if td_mask.sum() > 0:
            # TD subjects should have scale close to 1 (no transform)
            td_scale = scale[td_mask].mean(dim=0)
            td_reg = (td_scale - 1.0).pow(2).mean()
        else:
            td_reg = torch.tensor(0.0).to(device)
        
        loss = cls_loss + 0.1 * td_reg - 0.01 * scale_reg
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = (asd_prob.squeeze() > 0.5).float()
        train_correct += (preds == batch_y).sum().item()
        train_total += len(batch_y)
    
    # Validate
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            _, _, asd_prob = model(batch_x)
            preds = (asd_prob.squeeze() > 0.5).float()
            val_correct += (preds == batch_y).sum().item()
            val_total += len(batch_y)
    
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    scheduler.step(1 - val_acc)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        torch.save({
            'model_state_dict': model.state_dict(),
            'val_acc': val_acc,
            'epoch': epoch,
            'asd_mean_conn': asd_mean_conn.cpu(),
            'td_mean_conn': td_mean_conn.cpu(),
        }, 'neurodiverse_transform.pt')
    
    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d} | Loss: {train_loss/len(train_loader):.4f} | '
              f'Train: {train_acc:.1%} | Val: {val_acc:.1%} | Best: {best_val_acc:.1%} (ep {best_epoch})')

print(f'\nBest validation accuracy: {best_val_acc:.1%} at epoch {best_epoch}')
print(f'Model saved to neurodiverse_transform.pt')

## Step 4: Test the Transform

In [ ]:
# Load best model
checkpoint = torch.load('neurodiverse_transform.pt')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f'Loaded model from epoch {checkpoint["epoch"]} (val acc: {checkpoint["val_acc"]:.1%})')

# Get mean ASD connectivity (this is what we'll use for inference)
asd_conn = checkpoint['asd_mean_conn'].unsqueeze(0).to(device)

# Create a fake NT brain prediction to test
fake_nt = torch.randn(1, 20484).to(device)
with torch.no_grad():
    nd_pred = model.transform_brain(fake_nt, asd_conn)

diff = (nd_pred - fake_nt).abs().cpu().numpy().squeeze()
print(f'Mean absolute transform: {diff.mean():.4f}')
print(f'Max transform: {diff.max():.4f}')
print(f'Vertices changed > 0.1: {(diff > 0.1).sum()} / {len(diff)}')

## Step 5: Download Checkpoint

Download `neurodiverse_transform.pt` and upload it to the Azure VM.

Or run the cell below to push it directly.

In [ ]:
# Option A: Download to your computer
from google.colab import files
files.download('neurodiverse_transform.pt')

In [ ]:
# Option B: Push to HuggingFace Hub (then VM can download it)
from huggingface_hub import HfApi
api = HfApi()

# Create repo if needed
try:
    api.create_repo('Ibrhimovic9989/neurobrain-nd-transform', exist_ok=True)
except:
    pass

api.upload_file(
    path_or_fileobj='neurodiverse_transform.pt',
    path_in_repo='neurodiverse_transform.pt',
    repo_id='Ibrhimovic9989/neurobrain-nd-transform',
)
print('Uploaded to HuggingFace: Ibrhimovic9989/neurobrain-nd-transform')
print('VM can download with: huggingface_hub.hf_hub_download("Ibrhimovic9989/neurobrain-nd-transform", "neurodiverse_transform.pt")')